# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset package using the `mlcroissant` library, referencing entities strictly by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD):
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

**Dataset Description:**
> Structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility. Datasets include clearly labeled columns: (1) Table 1—age at diagnosis, height, and presence of clitomegaly, hirsutism, alopecia, acne, oligomenorrhea, and infertility; (2) Table 2—hormone levels, adrenal CT, and gynecological ultrasound findings; (3) Table 3—genetic variants of CYP21A2 with chromosomal location, variant details, population frequency, zygosity, pathogenicity assessment, and inferred phenotype.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print metadata about the dataset
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Keywords:", getattr(metadata, 'keywords', []))
print("Croissant Version:", getattr(metadata, 'conformsTo', None))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We list all the record sets, and for each record set, all fields and columns by their `@id`s.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets are defined in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if fields:
            print("  Field @ids:")
            for fld in fields:
                if isinstance(fld, dict) and '@id' in fld:
                    print(f"    - {fld['@id']}")
                else:
                    print(f"    - {fld}")
        columns = rs.get('column', [])
        if columns:
            print("  Column @ids:")
            for col in columns:
                if isinstance(col, dict) and '@id' in col:
                    print(f"    - {col['@id']}")
                else:
                    print(f"    - {col}")
        print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.

All record sets are referenced by their `@id`. Columns and fields are referenced by their `@id`s as well.

In [ ]:
# Extract data from each record set, using @id for access.
dataframes = {}

# List to keep visible record set @ids
record_set_ids = []

for rs in dataset.record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for RecordSet @id {rs_id}: {e}")

# Example: pick first record set for demonstration
if record_set_ids:
    first_rs_id = record_set_ids[0]
else:
    first_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalizing numeric fields, and categorizing.

Reference fields and columns strictly by their `@id`. If no numeric fields are available, adapt the section for categorical data.

In [ ]:
# For demonstration, use the first record set extracted above
import numpy as np

if first_rs_id and first_rs_id in dataframes:
    df = dataframes[first_rs_id]

    # Try to infer a numeric field from the DataFrame
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)

    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical column if available
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
        # Example: Show counts of unique values in a categorical column
        categorical_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if categorical_fields:
            cat_field = categorical_fields[0]
            value_counts = df[cat_field].value_counts()
            print(f"Counts of unique values for {cat_field}:")
            print(value_counts)
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This section references columns strictly by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_rs_id and first_rs_id in dataframes:
    df = dataframes[first_rs_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        # Histogram for the first numeric field
        field_id = numeric_fields[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df[field_id], bins=10, kde=True)
        plt.title(f'Distribution of {field_id} (@id)')
        plt.xlabel(field_id)
        plt.ylabel('Frequency')
        plt.show()

        # If there are two numeric fields, scatter plot
        if len(numeric_fields) > 1:
            field_id2 = numeric_fields[1]
            plt.figure(figsize=(6, 4))
            sns.scatterplot(x=df[field_id], y=df[field_id2])
            plt.title(f'{field_id} vs {field_id2} (@id)')
            plt.xlabel(field_id)
            plt.ylabel(field_id2)
            plt.show()
    else:
        # If no numeric fields, show barplot for counts of a categorical field
        cat_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if cat_fields:
            cat_field = cat_fields[0]
            plt.figure(figsize=(6, 4))
            sns.countplot(y=cat_field, data=df)
            plt.title(f'Counts of {cat_field} (@id)')
            plt.xlabel('Count')
            plt.ylabel(cat_field)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrates FAIR data access via the `mlcroissant` library referencing the Croissant schema strictly by `@id`. You explored tabular clinical, hormone, and genetic variant data for NC-21OHD-associated infertility in a rare disease cohort. Small sample and specialized data mean this resource is best for academic exploration and reanalysis.

- Data are loaded and interpreted strictly via record set, field, and column `@id`s.
- The cohort is highly limited (three patients, all female, all from the same site).
- The schema enables standardized access, but use caution extrapolating results.
- For more information, refer directly to the Croissant schema and documentation.